In [12]:
# ============================================================
# SMS PROBABILITY MODEL FOR OBIA MOUND CANDIDATES
# Object based ranking using magnetic anomaly, faults and backscatter
# Master grid is the 2 m bathymetry
# Fault evidence now includes proximity and fault density
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize, shapes

from scipy.ndimage import (
    label as connected_label,
    find_objects,
    binary_dilation,
    distance_transform_edt,
    uniform_filter
)

try:
    import geopandas as gpd
    from shapely.geometry import shape
    GEOPANDAS_AVAILABLE = True
except Exception:
    GEOPANDAS_AVAILABLE = False

warnings.filterwarnings("ignore")


# ============================================================
# 1. Paths
# ============================================================

data_dir = Path(r"")

bathymetry_path = data_dir / "bathymetry_2m_epsg32623_reproject.tif"

mound_candidate_path = data_dir / "label_assisted_candidates_final_letsgo_merged_id_2m.tif"
magnetic_path = data_dir / "Magnetic_anomaly_reproject.tif"
faults_path = data_dir / "faults_QA_model_ready.gpkg"
backscatter_path = data_dir / "backscatter 35.tif"

output_dir = data_dir / "sms_probability_model_outputs_trymound_new_merge"
output_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Main settings
# ============================================================

mound_probability_threshold = 0.50

minimum_mound_area_m2 = 250.0

magnetic_search_buffer_m = 45.0
backscatter_search_buffer_m = 55.0
fault_search_buffer_m = 35.0

background_inner_buffer_m = 120.0
background_outer_buffer_m = 350.0

magnetic_low_is_sms = True

backscatter_low_is_sms = True

fault_full_score_distance_m = 40.0
fault_zero_score_distance_m = 250.0

fault_density_radius_m = 150.0
fault_density_full_length_m = 250.0

fault_proximity_weight = 0.60
fault_density_weight = 0.40

weights = {
    "magnetic": 0.45,
    "fault": 0.35,
    "backscatter": 0.10,
    "morphology": 0.10
}

magnetic_z_start = 0.25
magnetic_z_full = 2.50

backscatter_z_start = 0.25
backscatter_z_full = 2.00

high_probability_threshold = 0.70
medium_probability_threshold = 0.45

probability_transform = "linear"
logistic_midpoint = 0.50
logistic_steepness = 6.00


# ============================================================
# 3. Helper functions
# ============================================================

def clean_output_profile(profile):
    out_profile = profile.copy()

    for key in [
        "blockxsize",
        "blockysize",
        "BLOCKXSIZE",
        "BLOCKYSIZE"
    ]:
        out_profile.pop(key, None)

    out_profile.update(
        driver="GTiff",
        tiled=True,
        blockxsize=256,
        blockysize=256,
        compress="deflate",
        BIGTIFF="IF_SAFER"
    )

    return out_profile


def write_float_raster(path, array, profile, nodata=-9999.0):
    out_profile = clean_output_profile(profile)

    out_profile.update(
        dtype="float32",
        count=1,
        nodata=nodata,
        predictor=3
    )

    out = np.where(
        np.isfinite(array),
        array,
        nodata
    ).astype("float32")

    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(out, 1)


def write_uint8_raster(path, array, profile):
    out_profile = clean_output_profile(profile)

    out_profile.update(
        dtype="uint8",
        count=1,
        nodata=0
    )

    out = np.where(array > 0, 1, 0).astype("uint8")

    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(out, 1)


def write_int32_raster(path, array, profile):
    out_profile = clean_output_profile(profile)

    out_profile.update(
        dtype="int32",
        count=1,
        nodata=0
    )

    out = np.where(array > 0, array, 0).astype("int32")

    with rasterio.open(path, "w", **out_profile) as dst:
        dst.write(out, 1)


def same_grid(src, master_profile):
    return (
        src.crs == master_profile["crs"]
        and src.width == master_profile["width"]
        and src.height == master_profile["height"]
        and src.transform.almost_equals(master_profile["transform"])
    )


def read_and_align_raster(raster_path, master_profile, resampling_method):
    if not raster_path.exists():
        raise FileNotFoundError(f"Missing raster: {raster_path}")

    with rasterio.open(raster_path) as src:
        source = src.read(1, masked=True).astype("float32").filled(np.nan)

        destination = np.full(
            (master_profile["height"], master_profile["width"]),
            np.nan,
            dtype="float32"
        )

        if same_grid(src, master_profile):
            destination = source.astype("float32")
            was_resampled = False
        else:
            reproject(
                source=source,
                destination=destination,
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=np.nan,
                dst_transform=master_profile["transform"],
                dst_crs=master_profile["crs"],
                dst_nodata=np.nan,
                resampling=resampling_method
            )
            was_resampled = True

    destination[~np.isfinite(destination)] = np.nan

    return destination, was_resampled


def robust_scale(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if values.size < 10:
        return np.nan

    median = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - median))
    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale <= 0:
        q16, q84 = np.nanpercentile(values, [16, 84])
        scale = (q84 - q16) / 2.0

    if not np.isfinite(scale) or scale <= 0:
        scale = np.nanstd(values)

    if not np.isfinite(scale) or scale <= 0:
        scale = np.nan

    return scale


def fuzzy_linear(value, start, full):
    if not np.isfinite(value):
        return 0.0

    if value <= start:
        return 0.0

    if value >= full:
        return 1.0

    return float((value - start) / (full - start))


def score_from_weighted_evidence(evidence_score):
    evidence_score = np.clip(evidence_score, 0.0, 1.0)

    if probability_transform.lower() == "logistic":
        return float(
            1.0 / (
                1.0 + np.exp(
                    -logistic_steepness * (evidence_score - logistic_midpoint)
                )
            )
        )

    return float(evidence_score)


def priority_label(probability):
    if probability >= high_probability_threshold:
        return "High"

    if probability >= medium_probability_threshold:
        return "Medium"

    return "Low"


def meters_to_pixels(distance_m, pixel_size_m):
    return max(1, int(np.ceil(distance_m / pixel_size_m)))


def expanded_slice(source_slice, pad_pixels, shape):
    row_slice, col_slice = source_slice

    row_start = max(0, row_slice.start - pad_pixels)
    row_stop = min(shape[0], row_slice.stop + pad_pixels)

    col_start = max(0, col_slice.start - pad_pixels)
    col_stop = min(shape[1], col_slice.stop + pad_pixels)

    return np.s_[row_start:row_stop, col_start:col_stop]


def finite_percentile(values, percentile):
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    return float(np.nanpercentile(values, percentile))


def finite_mean(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    return float(np.nanmean(values))


def finite_median(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    return float(np.nanmedian(values))


# ============================================================
# 4. Load 2 m master bathymetry
# ============================================================

if not bathymetry_path.exists():
    raise FileNotFoundError(
        "The 2 m bathymetry master raster was not found. "
        "Set bathymetry_path to your 2 m bathymetry file before running."
    )

with rasterio.open(bathymetry_path) as src:
    bathymetry = src.read(1, masked=True).astype("float32").filled(np.nan)
    master_profile = src.profile.copy()
    master_crs = src.crs
    master_transform = src.transform

valid_area = np.isfinite(bathymetry)

pixel_size_x = abs(master_transform.a)
pixel_size_y = abs(master_transform.e)
pixel_size = float((pixel_size_x + pixel_size_y) / 2.0)
pixel_area_m2 = pixel_size_x * pixel_size_y

print("Master bathymetry loaded")
print("CRS:", master_crs)
print("Size:", master_profile["width"], "x", master_profile["height"])
print("Pixel size:", round(pixel_size_x, 3), "x", round(pixel_size_y, 3), "m")


# ============================================================
# 5. Align candidate mask, magnetic and backscatter to 2 m grid
# ============================================================

mound_candidate, mound_resampled = read_and_align_raster(
    mound_candidate_path,
    master_profile,
    Resampling.nearest
)

magnetic, magnetic_resampled = read_and_align_raster(
    magnetic_path,
    master_profile,
    Resampling.bilinear
)

backscatter, backscatter_resampled = read_and_align_raster(
    backscatter_path,
    master_profile,
    Resampling.bilinear
)

write_float_raster(output_dir / "magnetic_aligned_to_2m.tif", magnetic, master_profile)
write_float_raster(output_dir / "backscatter_aligned_to_2m.tif", backscatter, master_profile)

print("Mound mask resampled:", mound_resampled)
print("Magnetic resampled:", magnetic_resampled)
print("Backscatter resampled:", backscatter_resampled)


# ============================================================
# 6. Create mound objects from part 1
# ============================================================

candidate_values = mound_candidate[np.isfinite(mound_candidate)]

if candidate_values.size == 0:
    raise ValueError("The mound candidate raster contains no valid values.")

candidate_min = float(np.nanmin(candidate_values))
candidate_max = float(np.nanmax(candidate_values))

if candidate_min >= 0 and candidate_max <= 1.0:
    mound_confidence_pixel = np.clip(mound_candidate, 0.0, 1.0)
    mound_mask = mound_confidence_pixel >= mound_probability_threshold
else:
    mound_confidence_pixel = np.where(mound_candidate > 0, 1.0, 0.0).astype("float32")
    mound_mask = mound_candidate > 0

mound_mask = mound_mask & valid_area

raw_labels, raw_count = connected_label(mound_mask)

object_slices = find_objects(raw_labels)
labels = np.zeros(raw_labels.shape, dtype="int32")

new_id = 0

for old_id, source_slice in enumerate(object_slices, start=1):
    if source_slice is None:
        continue

    object_mask_local = raw_labels[source_slice] == old_id
    pixel_count = int(object_mask_local.sum())
    area_m2 = pixel_count * pixel_area_m2

    if area_m2 < minimum_mound_area_m2:
        continue

    new_id += 1
    labels[source_slice][object_mask_local] = new_id

mound_count = new_id
mound_mask = labels > 0
object_slices = find_objects(labels)

if mound_count == 0:
    raise ValueError("No mound objects remain after area filtering.")

write_uint8_raster(output_dir / "mound_candidates_clean_2m.tif", mound_mask, master_profile)
write_int32_raster(output_dir / "mound_object_id_2m.tif", labels, master_profile)

print("Mound objects:", mound_count)


# ============================================================
# 7. Create fault proximity and fault density layers
# ============================================================

if not GEOPANDAS_AVAILABLE:
    raise ImportError(
        "geopandas and shapely are required to read the fault GeoPackage."
    )

if not faults_path.exists():
    raise FileNotFoundError(f"Missing faults file: {faults_path}")

faults = gpd.read_file(faults_path)

if faults.empty:
    raise ValueError("The fault GeoPackage is empty.")

faults = faults[~faults.geometry.is_empty & faults.geometry.notnull()].copy()

if faults.empty:
    raise ValueError("The fault GeoPackage has no valid geometries.")

if faults.crs is None:
    print("Fault CRS is missing. The code assumes the faults are already in the bathymetry CRS.")
    faults = faults.set_crs(master_crs)
else:
    faults = faults.to_crs(master_crs)

fault_shapes = [
    (geom, 1)
    for geom in faults.geometry
    if geom is not None and not geom.is_empty
]

fault_raster = rasterize(
    fault_shapes,
    out_shape=(master_profile["height"], master_profile["width"]),
    transform=master_transform,
    fill=0,
    default_value=1,
    dtype="uint8",
    all_touched=True
).astype(bool)

distance_to_faults = distance_transform_edt(
    ~fault_raster,
    sampling=(pixel_size_y, pixel_size_x)
).astype("float32")

distance_to_faults[~valid_area] = np.nan

fault_proximity_score_pixel = (
    (fault_zero_score_distance_m - distance_to_faults)
    / (fault_zero_score_distance_m - fault_full_score_distance_m)
)

fault_proximity_score_pixel = np.clip(fault_proximity_score_pixel, 0.0, 1.0)
fault_proximity_score_pixel[distance_to_faults <= fault_full_score_distance_m] = 1.0
fault_proximity_score_pixel[distance_to_faults >= fault_zero_score_distance_m] = 0.0
fault_proximity_score_pixel[~np.isfinite(distance_to_faults)] = np.nan

fault_density_radius_px = meters_to_pixels(fault_density_radius_m, pixel_size)
fault_density_window_size = fault_density_radius_px * 2 + 1

fault_pixel_length = fault_raster.astype("float32") * pixel_size

fault_length_density = uniform_filter(
    fault_pixel_length,
    size=fault_density_window_size,
    mode="constant",
    cval=0.0
) * (fault_density_window_size ** 2)

fault_length_density[~valid_area] = np.nan

fault_density_score_pixel = np.clip(
    fault_length_density / fault_density_full_length_m,
    0.0,
    1.0
)

fault_density_score_pixel[~valid_area] = np.nan

write_uint8_raster(output_dir / "fault_raster_2m.tif", fault_raster, master_profile)

write_float_raster(
    output_dir / "distance_to_faults_2m.tif",
    distance_to_faults,
    master_profile
)

write_float_raster(
    output_dir / "fault_proximity_score_2m.tif",
    fault_proximity_score_pixel,
    master_profile
)

write_float_raster(
    output_dir / "fault_length_density_150m_2m.tif",
    fault_length_density,
    master_profile
)

write_float_raster(
    output_dir / "fault_density_score_2m.tif",
    fault_density_score_pixel,
    master_profile
)

print("Fault proximity and fault density created")


# ============================================================
# 8. Global fallback statistics for magnetic and backscatter
# ============================================================

background_not_mound = valid_area & (~mound_mask)

global_magnetic_background = magnetic[background_not_mound & np.isfinite(magnetic)]
global_backscatter_background = backscatter[background_not_mound & np.isfinite(backscatter)]

global_magnetic_median = finite_median(global_magnetic_background)
global_backscatter_median = finite_median(global_backscatter_background)

global_magnetic_scale = robust_scale(global_magnetic_background)
global_backscatter_scale = robust_scale(global_backscatter_background)

if not np.isfinite(global_magnetic_scale):
    global_magnetic_scale = 1.0

if not np.isfinite(global_backscatter_scale):
    global_backscatter_scale = 1.0


# ============================================================
# 9. Object based evidence extraction
# ============================================================

records = []

magnetic_score_raster = np.full(labels.shape, np.nan, dtype="float32")
backscatter_score_raster = np.full(labels.shape, np.nan, dtype="float32")

fault_proximity_object_score_raster = np.full(labels.shape, np.nan, dtype="float32")
fault_density_object_score_raster = np.full(labels.shape, np.nan, dtype="float32")
fault_combined_object_score_raster = np.full(labels.shape, np.nan, dtype="float32")

morphology_score_raster = np.full(labels.shape, np.nan, dtype="float32")
sms_score_raster = np.full(labels.shape, np.nan, dtype="float32")
sms_probability_raster = np.full(labels.shape, np.nan, dtype="float32")

magnetic_search_px = meters_to_pixels(magnetic_search_buffer_m, pixel_size)
backscatter_search_px = meters_to_pixels(backscatter_search_buffer_m, pixel_size)
fault_search_px = meters_to_pixels(fault_search_buffer_m, pixel_size)

background_inner_px = meters_to_pixels(background_inner_buffer_m, pixel_size)
background_outer_px = meters_to_pixels(background_outer_buffer_m, pixel_size)

max_pad_px = max(
    magnetic_search_px,
    backscatter_search_px,
    fault_search_px,
    background_outer_px
) + 5

for mound_id, source_slice in enumerate(object_slices, start=1):
    if source_slice is None:
        continue

    local_slice = expanded_slice(source_slice, max_pad_px, labels.shape)

    local_labels = labels[local_slice]
    object_mask = local_labels == mound_id
    all_mounds_local = mound_mask[local_slice]

    object_pixel_count = int(object_mask.sum())
    object_area_m2 = object_pixel_count * pixel_area_m2
    equivalent_diameter_m = float(np.sqrt(4.0 * object_area_m2 / np.pi))

    morphology_values = mound_confidence_pixel[local_slice][object_mask]
    morphology_score = finite_mean(morphology_values)

    if not np.isfinite(morphology_score):
        morphology_score = 1.0

    morphology_score = float(np.clip(morphology_score, 0.0, 1.0))

    magnetic_zone = binary_dilation(object_mask, iterations=magnetic_search_px)
    backscatter_zone = binary_dilation(object_mask, iterations=backscatter_search_px)
    fault_zone = binary_dilation(object_mask, iterations=fault_search_px)

    ring_outer = binary_dilation(object_mask, iterations=background_outer_px)
    ring_inner = binary_dilation(object_mask, iterations=background_inner_px)

    local_background_ring = ring_outer & (~ring_inner) & (~all_mounds_local)

    magnetic_local = magnetic[local_slice]
    magnetic_object_values = magnetic_local[
        magnetic_zone & np.isfinite(magnetic_local)
    ]
    magnetic_background_values = magnetic_local[
        local_background_ring & np.isfinite(magnetic_local)
    ]

    if magnetic_background_values.size < 20:
        magnetic_background_median = global_magnetic_median
        magnetic_background_scale = global_magnetic_scale
    else:
        magnetic_background_median = finite_median(magnetic_background_values)
        magnetic_background_scale = robust_scale(magnetic_background_values)

        if not np.isfinite(magnetic_background_scale):
            magnetic_background_scale = global_magnetic_scale

    if magnetic_low_is_sms:
        magnetic_object_indicator = finite_percentile(magnetic_object_values, 20)
        magnetic_contrast = magnetic_background_median - magnetic_object_indicator
    else:
        magnetic_object_indicator = finite_percentile(magnetic_object_values, 80)
        magnetic_contrast = magnetic_object_indicator - magnetic_background_median

    if np.isfinite(magnetic_contrast):
        magnetic_z = magnetic_contrast / magnetic_background_scale
    else:
        magnetic_z = np.nan

    magnetic_score = fuzzy_linear(
        magnetic_z,
        magnetic_z_start,
        magnetic_z_full
    )

    backscatter_local = backscatter[local_slice]
    backscatter_object_values = backscatter_local[
        backscatter_zone & np.isfinite(backscatter_local)
    ]
    backscatter_background_values = backscatter_local[
        local_background_ring & np.isfinite(backscatter_local)
    ]

    if backscatter_background_values.size < 20:
        backscatter_background_median = global_backscatter_median
        backscatter_background_scale = global_backscatter_scale
    else:
        backscatter_background_median = finite_median(backscatter_background_values)
        backscatter_background_scale = robust_scale(backscatter_background_values)

        if not np.isfinite(backscatter_background_scale):
            backscatter_background_scale = global_backscatter_scale

    if backscatter_low_is_sms:
        backscatter_object_indicator = finite_percentile(backscatter_object_values, 20)
        backscatter_contrast = backscatter_background_median - backscatter_object_indicator
    else:
        backscatter_object_indicator = finite_percentile(backscatter_object_values, 80)
        backscatter_contrast = backscatter_object_indicator - backscatter_background_median

    if np.isfinite(backscatter_contrast):
        backscatter_z = backscatter_contrast / backscatter_background_scale
    else:
        backscatter_z = np.nan

    backscatter_score = fuzzy_linear(
        backscatter_z,
        backscatter_z_start,
        backscatter_z_full
    )

    fault_distance_values = distance_to_faults[local_slice][fault_zone]
    fault_proximity_values = fault_proximity_score_pixel[local_slice][fault_zone]
    fault_density_values = fault_density_score_pixel[local_slice][fault_zone]
    fault_length_values = fault_length_density[local_slice][fault_zone]

    fault_proximity_score = finite_percentile(fault_proximity_values, 90)
    fault_density_score = finite_percentile(fault_density_values, 90)

    fault_distance_p05_m = finite_percentile(fault_distance_values, 5)
    fault_length_nearby_m = finite_percentile(fault_length_values, 90)

    if not np.isfinite(fault_proximity_score):
        fault_proximity_score = 0.0

    if not np.isfinite(fault_density_score):
        fault_density_score = 0.0

    fault_proximity_score = float(np.clip(fault_proximity_score, 0.0, 1.0))
    fault_density_score = float(np.clip(fault_density_score, 0.0, 1.0))

    fault_score = (
        fault_proximity_weight * fault_proximity_score
        + fault_density_weight * fault_density_score
    )

    fault_score = float(np.clip(fault_score, 0.0, 1.0))

    evidence_score = (
        weights["magnetic"] * magnetic_score
        + weights["fault"] * fault_score
        + weights["backscatter"] * backscatter_score
        + weights["morphology"] * morphology_score
    )

    evidence_score = float(np.clip(evidence_score, 0.0, 1.0))
    sms_probability = score_from_weighted_evidence(evidence_score)

    priority = priority_label(sms_probability)

    rows, cols = np.where(labels == mound_id)

    magnetic_score_raster[rows, cols] = magnetic_score
    backscatter_score_raster[rows, cols] = backscatter_score

    fault_proximity_object_score_raster[rows, cols] = fault_proximity_score
    fault_density_object_score_raster[rows, cols] = fault_density_score
    fault_combined_object_score_raster[rows, cols] = fault_score

    morphology_score_raster[rows, cols] = morphology_score
    sms_score_raster[rows, cols] = evidence_score
    sms_probability_raster[rows, cols] = sms_probability

    records.append(
        {
            "mound_id": mound_id,
            "area_m2": object_area_m2,
            "equiv_diam_m": equivalent_diameter_m,

            "morph_score": morphology_score,

            "mag_score": magnetic_score,
            "mag_z": magnetic_z,
            "mag_obj": magnetic_object_indicator,
            "mag_bg": magnetic_background_median,
            "mag_contr": magnetic_contrast,

            "bs_score": backscatter_score,
            "bs_z": backscatter_z,
            "bs_obj": backscatter_object_indicator,
            "bs_bg": backscatter_background_median,
            "bs_contr": backscatter_contrast,

            "fault_score": fault_score,
            "fault_prox_score": fault_proximity_score,
            "fault_density_score": fault_density_score,
            "fault_p05_m": fault_distance_p05_m,
            "fault_length_nearby_m": fault_length_nearby_m,

            "sms_score": evidence_score,
            "sms_prob": sms_probability,
            "priority": priority
        }
    )


# ============================================================
# 10. Export raster outputs
# ============================================================

write_float_raster(
    output_dir / "magnetic_score_objects_2m.tif",
    magnetic_score_raster,
    master_profile
)

write_float_raster(
    output_dir / "backscatter_score_objects_2m.tif",
    backscatter_score_raster,
    master_profile
)

write_float_raster(
    output_dir / "fault_proximity_score_objects_2m.tif",
    fault_proximity_object_score_raster,
    master_profile
)

write_float_raster(
    output_dir / "fault_density_score_objects_2m.tif",
    fault_density_object_score_raster,
    master_profile
)

write_float_raster(
    output_dir / "fault_combined_score_objects_2m.tif",
    fault_combined_object_score_raster,
    master_profile
)

write_float_raster(
    output_dir / "morphology_score_objects_2m.tif",
    morphology_score_raster,
    master_profile
)

write_float_raster(
    output_dir / "sms_evidence_score_2m.tif",
    sms_score_raster,
    master_profile
)

write_float_raster(
    output_dir / "sms_probability_2m.tif",
    sms_probability_raster,
    master_profile
)

print("Raster outputs written")


# ============================================================
# 11. Export CSV and GeoPackage object table
# ============================================================

results = pd.DataFrame(records)

results = results.sort_values(
    by=[
        "sms_prob",
        "sms_score",
        "mag_score",
        "fault_score",
        "fault_density_score"
    ],
    ascending=False
).reset_index(drop=True)

results["rank"] = np.arange(1, len(results) + 1)

geometries = []
geometry_ids = []

for geom, value in shapes(
    labels.astype("int32"),
    mask=labels > 0,
    transform=master_transform
):
    mound_id = int(value)

    if mound_id <= 0:
        continue

    geometries.append(shape(geom))
    geometry_ids.append(mound_id)

geometry_table = gpd.GeoDataFrame(
    {"mound_id": geometry_ids},
    geometry=geometries,
    crs=master_crs
)

geometry_table = geometry_table.dissolve(
    by="mound_id",
    as_index=False
)

output_gdf = geometry_table.merge(results, on="mound_id", how="left")

fault_count_buffer_m = fault_density_radius_m

fault_counts = []
fault_total_lengths = []

faults_for_count = faults.copy()
faults_for_count["fault_length_m"] = faults_for_count.geometry.length

for geom in output_gdf.geometry:
    search_geom = geom.buffer(fault_count_buffer_m)

    possible_idx = list(faults_for_count.sindex.intersection(search_geom.bounds))
    possible_faults = faults_for_count.iloc[possible_idx].copy()

    if possible_faults.empty:
        fault_counts.append(0)
        fault_total_lengths.append(0.0)
        continue

    selected_faults = possible_faults[possible_faults.intersects(search_geom)]

    fault_counts.append(int(len(selected_faults)))
    fault_total_lengths.append(float(selected_faults.geometry.intersection(search_geom).length.sum()))

output_gdf["fault_count_150m"] = fault_counts
output_gdf["fault_vector_length_150m"] = fault_total_lengths

results = results.merge(
    output_gdf[
        [
            "mound_id",
            "fault_count_150m",
            "fault_vector_length_150m"
        ]
    ],
    on="mound_id",
    how="left"
)

results = results.sort_values(
    by=[
        "sms_prob",
        "sms_score",
        "mag_score",
        "fault_score",
        "fault_density_score"
    ],
    ascending=False
).reset_index(drop=True)

results["rank"] = np.arange(1, len(results) + 1)

csv_path = output_dir / "sms_mound_probability_table.csv"
results.to_csv(csv_path, index=False)

output_gdf = geometry_table.merge(results, on="mound_id", how="left")

gpkg_path = output_dir / "sms_mound_probability_objects.gpkg"

output_gdf.to_file(
    gpkg_path,
    layer="sms_probability_objects",
    driver="GPKG"
)

print("GeoPackage written:", gpkg_path)
print("CSV written:", csv_path)


# ============================================================
# 12. Print summary
# ============================================================

print("")
print("Top 15 mound targets")

print(
    results[
        [
            "rank",
            "mound_id",
            "sms_prob",
            "priority",
            "mag_score",
            "fault_score",
            "fault_prox_score",
            "fault_density_score",
            "fault_count_150m",
            "fault_vector_length_150m",
            "bs_score",
            "morph_score",
            "area_m2",
            "equiv_diam_m",
            "fault_p05_m"
        ]
    ].head(15).to_string(index=False)
)

print("")
print("Finished")
print("Main outputs:")
print(output_dir / "sms_probability_2m.tif")
print(output_dir / "sms_evidence_score_2m.tif")
print(output_dir / "fault_proximity_score_2m.tif")
print(output_dir / "fault_density_score_2m.tif")
print(output_dir / "fault_combined_score_objects_2m.tif")
print(output_dir / "sms_mound_probability_objects.gpkg")
print(output_dir / "sms_mound_probability_table.csv")

Master bathymetry loaded
CRS: EPSG:32623
Size: 4985 x 5067
Pixel size: 1.888 x 1.888 m
Mound mask resampled: True
Magnetic resampled: True
Backscatter resampled: True
Mound objects: 306
Fault proximity and fault density created
Raster outputs written
GeoPackage written: sms_probability_model_outputs_trymound_new_merge\sms_mound_probability_objects.gpkg
CSV written: sms_probability_model_outputs_trymound_new_merge\sms_mound_probability_table.csv

Top 15 mound targets
 rank  mound_id  sms_prob priority  mag_score  fault_score  fault_prox_score  fault_density_score  fault_count_150m  fault_vector_length_150m  bs_score  morph_score       area_m2  equiv_diam_m  fault_p05_m
    1        78  1.000000     High   1.000000     1.000000          1.000000                  1.0                19               2702.226970  1.000000          1.0  29898.443917    195.109921     9.923924
    2       209  0.992730     High   1.000000     1.000000          1.000000                  1.0                 2  